# Tutor only: Week 3 exercise solutions

Every cell below has been run, so the outputs are the correct answers. Accept other correct approaches.

**Project scenario: Which children are being left behind?**

After last week's outreach review, the district ran a household survey. Each row of `immunisation_households.csv` is one household with a child aged 12 to 23 months.
The district health officer wants a model that flags households where the child is **unlikely to be fully immunised**, so outreach teams can visit them first. You are the analyst.

All of this data is made up.

| Column | Meaning |
|---|---|
| `household_id` | Unique ID of the household |
| `district` | District the household is in |
| `mother_age` | Mother's age in years |
| `mother_education` | No schooling, Primary, Secondary or Tertiary (sometimes not recorded) |
| `num_children` | Children in the household |
| `distance_km` | Distance to the nearest clinic (sometimes not recorded) |
| `household_income` | Monthly household income in dollars (sometimes not stated) |
| `birth_place` | Where the child was born: Facility or Home |
| `outreach_visits` | Outreach sessions held near the household this year |
| `fully_immunised` | **Target.** 1 = the child is fully immunised, 0 = not |

---
## Part A · Handling missing data and preparing features

### Exercise 1  *(3 marks)*

Import pandas. Load `immunisation_households.csv` into a DataFrame called `immun_df`. Print its shape, display the first 5 rows, and print the percentage of children who are fully immunised (1 decimal place).

In [1]:
import pandas as pd

immun_df = pd.read_csv("immunisation_households.csv")
print(immun_df.shape)
print("Fully immunised:", round(immun_df["fully_immunised"].mean() * 100, 1), "%")
immun_df.head()


(520, 10)
Fully immunised: 48.8 %


,household_id,district,mother_age,mother_education,num_children,distance_km,household_income,birth_place,outreach_visits,fully_immunised
0,H001,North,21,Primary,3,3.2,65.0,Facility,1,0
1,H002,South,22,Secondary,2,12.1,180.0,Facility,2,1
2,H003,North,26,Secondary,5,2.9,165.0,Facility,3,1
3,H004,South,30,Secondary,1,5.5,250.0,Facility,2,1
4,H005,North,22,Secondary,4,12.3,355.0,Facility,2,1


**Note:** The mean of a 0/1 column is the share of 1s. 1 mark each for loading, shape/head and the percentage.

### Exercise 2  *(4 marks)*

Show the number **and** the percentage of missing values for each column that has any. Then print how many rows have at least one missing value.

In [2]:
missing = immun_df.isna().sum()
percent = (immun_df.isna().mean() * 100).round(1)
print(missing[missing > 0])
print(percent[percent > 0])
print("Rows with at least one gap:", immun_df.isna().any(axis=1).sum())


mother_education    28
distance_km         48
household_income    62
dtype: int64
mother_education     5.4
distance_km          9.2
household_income    11.9
dtype: float64
Rows with at least one gap: 118


**Note:** `isna().sum()` counts, `isna().mean()` gives the share, and `any(axis=1)` checks across each row.

### Exercise 3  *(4 marks)*

If you dropped every row with a missing value, how many rows would be left, and what percentage of the table would be lost? In a comment, say whether you would drop them.

In [3]:
kept = immun_df.dropna()
lost = len(immun_df) - len(kept)
print("Rows left:", len(kept))
print("Lost:", lost, "rows, or", round(lost / len(immun_df) * 100, 1), "% of the table")
# Losing more than a fifth of the households is too costly, so I would fill the gaps instead.


Rows left: 402
Lost: 118 rows, or 22.7 % of the table


**Note:** 2 marks for the numbers, 2 for a reasoned comment. Accept 'drop' only if the student justifies it.

### Exercise 4  *(5 marks)*

Print the mean and the median of `household_income`. Then create a copy of the table called `clean_df` and fill the gaps in `household_income` and `distance_km` with the better of the two measures. In a comment, say why you chose it.

In [4]:
print("Mean:", round(immun_df["household_income"].mean(), 1))
print("Median:", immun_df["household_income"].median())

clean_df = immun_df.copy()
clean_df["household_income"] = clean_df["household_income"].fillna(clean_df["household_income"].median())
clean_df["distance_km"] = clean_df["distance_km"].fillna(clean_df["distance_km"].median())
print(clean_df[["household_income", "distance_km"]].isna().sum())
# The mean is pulled up by a few high incomes, so the median is the more typical value.


Mean: 186.9
Median: 155.0
household_income    0
distance_km         0
dtype: int64


**Note:** 1 mark for the comparison, 1 for working on a copy, 2 for the two fills, 1 for the reason (skewed data, so median).

### Exercise 5  *(4 marks)*

Fill the gaps in `mother_education` in `clean_df` with the most common value. Confirm that `clean_df` now has no missing values at all.

In [5]:
most_common = clean_df["mother_education"].mode()[0]
print("Most common:", most_common)
clean_df["mother_education"] = clean_df["mother_education"].fillna(most_common)
print("Gaps left:", clean_df.isna().sum().sum())


Most common: Secondary
Gaps left: 0


**Note:** `mode()[0]` gives the most frequent value. Also accept filling with a new label such as "Unknown", if the student explains the choice.

### Exercise 6  *(5 marks)*

Create a feature `far_from_clinic` in `clean_df` that is 1 when `distance_km` is greater than 10 and 0 otherwise. Compare the immunisation rate of the two groups (2 decimal places). In a comment, say whether this feature looks useful.

In [6]:
clean_df["far_from_clinic"] = (clean_df["distance_km"] > 10).astype(int)
print(clean_df.groupby("far_from_clinic")["fully_immunised"].mean().round(2))
# Children far from a clinic are much less likely to be fully immunised, so the feature carries a clue.


far_from_clinic
0    0.54
1    0.39
Name: fully_immunised, dtype: float64


**Note:** 2 marks for the feature, 2 for the groupby check, 1 for the conclusion.

### Exercise 7  *(5 marks)*

Encode the text columns of `clean_df`. (a) Turn `birth_place` into 1 for Facility and 0 for Home. (b) Education has a natural order, so turn `mother_education` into 0, 1, 2, 3 for No schooling, Primary, Secondary, Tertiary. (c) One-hot encode `district` with `pd.get_dummies`, storing the result in `ready_df`. Print the column names of `ready_df`.

In [7]:
clean_df["birth_place"] = clean_df["birth_place"].map({"Facility": 1, "Home": 0})
clean_df["mother_education"] = clean_df["mother_education"].map({"No schooling": 0, "Primary": 1, "Secondary": 2, "Tertiary": 3})
ready_df = pd.get_dummies(clean_df, columns=["district"], dtype=int)
print(list(ready_df.columns))


['household_id', 'mother_age', 'mother_education', 'num_children', 'distance_km', 'household_income', 'birth_place', 'outreach_visits', 'fully_immunised', 'far_from_clinic', 'district_East', 'district_North', 'district_South', 'district_West']


**Note:** Numbering is right for education because the levels have an order. It would be wrong for district, which has none: that's why district is one-hot encoded.

### Exercise 8  *(5 marks)*

Use `StandardScaler` to scale the columns `mother_age`, `num_children`, `distance_km`, `household_income` and `outreach_visits` of `ready_df`. Store the result in a copy called `scaled_df`, and show that each scaled column has a mean of about 0 and a standard deviation of about 1.

In [8]:
from sklearn.preprocessing import StandardScaler

num_cols = ["mother_age", "num_children", "distance_km", "household_income", "outreach_visits"]
scaled_df = ready_df.copy()
scaled_df[num_cols] = StandardScaler().fit_transform(ready_df[num_cols])
print(scaled_df[num_cols].describe().loc[["mean", "std"]].round(1))


      mother_age  num_children  distance_km  household_income  outreach_visits
mean         0.0          -0.0         -0.0              -0.0              0.0
std          1.0           1.0          1.0               1.0              1.0


**Note:** A mean shown as -0.0 is zero with a tiny rounding leftover. Full marks either way.

### Exercise 9  *(5 marks)*

Create a `SimpleImputer` with the median strategy and fit it on the **original** `distance_km` and `household_income` columns of `immun_df`. Print the values it learned. Then use it to fill a new household where both values are missing.

In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(immun_df[["distance_km", "household_income"]])
print("Learned:", imputer.statistics_)

new_home = pd.DataFrame({"distance_km": [None], "household_income": [None]}, dtype=float)
print(imputer.transform(new_home))


Learned: [  8.4 155. ]
[[  8.4 155. ]]


**Note:** 2 marks for fit, 1 for `statistics_`, 2 for transforming the new household. The learned values match the medians used in Exercise 4.

---
## Part B · Cross-validation and ML pipelines

In [10]:
import pandas as pd

immun_df = pd.read_csv("immunisation_households.csv")
X = immun_df.drop(columns=["household_id", "fully_immunised"])
y = immun_df["fully_immunised"]

num_cols = ["mother_age", "num_children", "distance_km", "household_income", "outreach_visits"]
cat_cols = ["district", "mother_education", "birth_place"]


### Exercise 1  *(4 marks)*

Split `X` and `y` into a training set and a test set. Keep 20% for testing, use `random_state=42` and stratify on `y`. Print the shape of each part and the immunisation rate in each.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Rate, train:", round(y_train.mean() * 100, 1), "%  test:", round(y_test.mean() * 100, 1), "%")


Train: (416, 8)  Test: (104, 8)
Rate, train: 48.8 %  test: 49.0 %


**Marking note:** 1 mark for each of: the four names in the right order, test_size, stratify, the printed check.

### Exercise 2  *(3 marks)*

What accuracy does a lazy model get on the test set if it always predicts the most common answer? Use `DummyClassifier`.

In [12]:
from sklearn.dummy import DummyClassifier

lazy = DummyClassifier(strategy="most_frequent")
lazy.fit(X_train, y_train)
print("Baseline:", round(lazy.score(X_test, y_test) * 100, 1), "%")


Baseline: 51.0 %


**Marking note:** This is the number every later model has to beat.

### Exercise 3  *(5 marks)*

Build a `Pipeline` that fills gaps with the median, scales, then fits a `KNeighborsClassifier`. Run 5-fold cross-validation on the **number columns** of the training set. Print the five scores, their mean and their standard deviation.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

num_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", KNeighborsClassifier()),
])
scores = cross_val_score(num_pipe, X_train[num_cols], y_train, cv=5)
print((scores * 100).round(1))
print("Mean:", round(scores.mean() * 100, 1), "±", round(scores.std() * 100, 1))


[53.6 61.4 55.4 57.8 54.2]
Mean: 56.5 ± 2.9


**Marking note:** 3 marks for the three steps in the right order, 2 for the cross-validation and summary.

### Exercise 4  *(6 marks)*

Build a `ColumnTransformer` called `prep` that sends the number columns through median filling and scaling, and the category columns through most-frequent filling and one-hot encoding (with `handle_unknown="ignore"`). Fit and transform the training set, then print its shape before and after.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numbers = Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categories = Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", numbers, num_cols), ("cat", categories, cat_cols)])

prepared = prep.fit_transform(X_train)
print("Before:", X_train.shape, " After:", prepared.shape)


Before: (416, 8)  After: (416, 15)


**Marking note:** 2 marks per branch, 1 for joining them, 1 for the shape check. 8 columns become 15: 5 numbers plus 4 districts, 4 education levels and 2 birth places.

### Exercise 5  *(4 marks)*

Join `prep` and a `KNeighborsClassifier` into one pipeline and cross-validate it (5 folds) on the full training set. Did the category columns improve on Exercise 3?

In [15]:
knn = Pipeline([("prep", prep), ("model", KNeighborsClassifier())])
scores = cross_val_score(knn, X_train, y_train, cv=5)
print("All columns, KNN:", round(scores.mean() * 100, 1), "±", round(scores.std() * 100, 1))
# Compare this mean with Exercise 3, keeping the ± in mind.


All columns, KNN: 59.6 ± 3.6


**Marking note:** Reward students who use the ± when they compare, instead of only looking at the means.

### Exercise 6  *(6 marks)*

Using the same `prep` and the same 5 folds, compare `KNeighborsClassifier`, `LogisticRegression(max_iter=1000)` and `DecisionTreeClassifier(random_state=42)`. Print the mean and standard deviation for each. Which would you choose?

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "KNN": KNeighborsClassifier(),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Decision tree": DecisionTreeClassifier(random_state=42),
}
for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5)
    print(name, "→", round(scores.mean() * 100, 1), "±", round(scores.std() * 100, 1))


KNN → 59.6 ± 3.6


Logistic regression → 67.1 ± 4.1


Decision tree → 56.5 ± 2.4


**Marking note:** 4 marks for a correct, fair comparison, 2 for a choice backed by the numbers.

### Exercise 7  *(7 marks)*

Fit your chosen pipeline on the whole training set and score it **once** on the test set. Then predict the chance of being fully immunised for the two households in the cell below (copy them into a DataFrame called `new_homes`). In a comment, tell the officer which household the outreach team should visit first, and how far the model can be trusted.

```python
new_homes = pd.DataFrame({
    "district": ["East", "North"],
    "mother_age": [19, 31],
    "mother_education": [None, "Secondary"],
    "num_children": [5, 2],
    "distance_km": [18.0, 3.5],
    "household_income": [None, 240],
    "birth_place": ["Home", "Facility"],
    "outreach_visits": [0, 4],
})
```

In [17]:
best = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
best.fit(X_train, y_train)
print("Test score:", round(best.score(X_test, y_test) * 100, 1), "%")

new_homes = pd.DataFrame({
    "district": ["East", "North"],
    "mother_age": [19, 31],
    "mother_education": [None, "Secondary"],
    "num_children": [5, 2],
    "distance_km": [18.0, 3.5],
    "household_income": [None, 240],
    "birth_place": ["Home", "Facility"],
    "outreach_visits": [0, 4],
})
print(best.predict_proba(new_homes)[:, 1].round(2))
# Visit the first household (East) first: its chance of full immunisation is far lower.
# The model beats the lazy baseline, but it is still wrong for roughly 3 households in 10,
# so it should guide the visiting order, not replace the team's judgement.


Test score: 72.1 %
[0.1  0.88]


**Marking note:** 2 marks for fit and a single test score, 2 for the predictions (the pipeline must handle the gaps), 3 for a recommendation that follows from the numbers and mentions the model's limits.